In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
from io import StringIO

# Load the dataset from the provided CSV content
csv_content = """carat,cut,color,clarity,depth,table,price,x,y,z
0.23,Ideal,E,SI2,61.5,55,326,3.95,3.98,2.43
0.21,Premium,E,SI1,59.8,61,326,3.89,3.84,2.31
..."""  # truncated for brevity; the full content is loaded in real execution
# For this solution we assume the full data is read as 'df'
df = pd.read_csv('diamonds.csv')
df.head()
# ========== 1. IQR outlier detection on price ==========
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
iqr_outliers = (df['price'] < lower_bound) | (df['price'] > upper_bound)
print(f"IQR flagged outliers: {iqr_outliers.sum()}")

# ========== 2. Isolation Forest with contamination=0.02 ==========
iso_forest = IsolationForest(contamination=0.02, random_state=42)
iso_pred = iso_forest.fit_predict(df[['price']])
if_outliers = iso_pred == -1
print(f"Isolation Forest flagged outliers: {if_outliers.sum()}")

# Overlap between the two methods
overlap = (iqr_outliers & if_outliers).sum()
print(f"Overlap count: {overlap}")
print(f"Overlap proportion (relative to IQR): {overlap / iqr_outliers.sum():.2%}")
print(f"Overlap proportion (relative to IF): {overlap / if_outliers.sum():.2%}")

# ========== 3. Winsorize price at 1st and 99th percentiles ==========
p01 = df['price'].quantile(0.01)
p99 = df['price'].quantile(0.99)
df['price_wins'] = df['price'].clip(lower=p01, upper=p99)

# Prepare features (carat + cut, color, clarity as categorical)
X = df[['carat', 'cut', 'color', 'clarity']]
y_orig = df['price']
y_wins = df['price_wins']

# Train/test split
X_train, X_test, y_orig_train, y_orig_test, y_wins_train, y_wins_test = train_test_split(
    X, y_orig, y_wins, test_size=0.2, random_state=42
)

# Preprocessor for categorical variables
categorical_features = ['cut', 'color', 'clarity']
preprocessor = ColumnTransformer(
    transformers=[('cat', OneHotEncoder(drop='first'), categorical_features)],
    remainder='passthrough'
)

# Model 1: linear regression on original price
pipeline_orig = Pipeline(steps=[('pre', preprocessor), ('reg', LinearRegression())])
pipeline_orig.fit(X_train, y_orig_train)
y_orig_pred = pipeline_orig.predict(X_test)
r2_orig = r2_score(y_orig_test, y_orig_pred)

# Model 2: linear regression on winsorized price
pipeline_wins = Pipeline(steps=[('pre', preprocessor), ('reg', LinearRegression())])
pipeline_wins.fit(X_train, y_wins_train)
y_wins_pred = pipeline_wins.predict(X_test)
r2_wins = r2_score(y_wins_test, y_wins_pred)

print(f"R² before winsorizing (on original price): {r2_orig:.4f}")
print(f"R² after winsorizing (on winsorized price): {r2_wins:.4f}")
print(f"Improvement: {r2_wins - r2_orig:.4f}")

# ========== 4. Duplicate injection and removal ==========
print(f"Original dataset shape: {df.shape}")
# Randomly select 50 rows to duplicate (with replacement)
np.random.seed(42)
duplicate_indices = np.random.choice(df.index, size=50, replace=True)
df_duplicated = pd.concat([df, df.loc[duplicate_indices]], ignore_index=True)
print(f"After injecting 50 duplicates: {df_duplicated.shape}")

# Deduplicate based on all columns
df_deduplicated = df_duplicated.drop_duplicates()
print(f"After dropping duplicates: {df_deduplicated.shape}")

if df_deduplicated.shape[0] == df.shape[0]:
    print("Success: All 50 artificial duplicates removed, back to original size.")
else:
    print("Mismatch – check deduplication logic.")

IQR flagged outliers: 3540
Isolation Forest flagged outliers: 1071
Overlap count: 1071
Overlap proportion (relative to IQR): 30.25%
Overlap proportion (relative to IF): 100.00%
R² before winsorizing (on original price): 0.9153
R² after winsorizing (on winsorized price): 0.9172
Improvement: 0.0019
Original dataset shape: (53940, 11)
After injecting 50 duplicates: (53990, 11)
After dropping duplicates: (53794, 11)
Mismatch – check deduplication logic.
